# 第134章: 時間的一貫性の技術 — 滑らかな動画生成のために

## 📋 この章で学ぶこと

この章を終えると、以下ができるようになります：

- [ ] フリッカー（ちらつき）とドリフト（漂流）の問題を理解し可視化できる
- [ ] Temporal Super-Resolution で疎なキーフレームから滑らかな動画を補間できる
- [ ] Autoregressive Video Sampler でフレームを逐次生成できる
- [ ] FVD（Fréchet Video Distance）の概念を説明できる
- [ ] 時間的一貫性メトリクスを実装し、手法間の比較ができる

## 🎯 前提知識

- ✅ Notebook 131（Video Diffusion Models）
- ✅ Notebook 132（Diffusion Transformer）

⏱️ **推定学習時間**: 120-150分
📊 **難易度**: ★★★★☆（上級）
🎓 **カテゴリ**: 時空間モデリング

## 目次

1. [時間的一貫性の問題 — フリッカーとドリフト](#section1)
2. [時間的一貫性メトリクス](#section2)
3. [Temporal Super-Resolution](#section3)
4. [Autoregressive Video Extension](#section4)
5. [実践的テクニック](#section5)
6. [比較実験](#section6)
7. [まとめと自己評価クイズ](#summary)

In [ ]:
# ============================================================
# 環境設定
# ============================================================
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.font_manager as fm
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
import warnings
import math

warnings.filterwarnings('ignore')

def setup_japanese_font():
    """日本語フォントを設定する"""
    japanese_fonts = [
        'Hiragino Sans', 'Hiragino Maru Gothic Pro',
        'Yu Gothic', 'MS Gothic',
        'Noto Sans CJK JP', 'IPAexGothic',
    ]
    available_fonts = set(f.name for f in fm.fontManager.ttflist)
    for font in japanese_fonts:
        if font in available_fonts:
            plt.rcParams['font.family'] = font
            plt.rcParams['axes.unicode_minus'] = False
            return font
    return None

font_used = setup_japanese_font()
plt.rcParams['figure.figsize'] = (12, 8)
plt.rcParams['font.size'] = 12

# 再現性の確保
torch.manual_seed(42)
np.random.seed(42)

# デバイス設定
device = torch.device('cuda' if torch.cuda.is_available() else
                       'mps' if torch.backends.mps.is_available() else 'cpu')
print(f'✅ ライブラリのインポート完了')
print(f'🖥️ デバイス: {device}')
print(f'📝 日本語フォント: {font_used}')

<a id="section1"></a>
## 1. 時間的一貫性の問題 — フリッカーとドリフト

### 🤔 動画生成で何が問題になるのか？

画像生成モデル（DDPMやStable Diffusionなど）は1枚ずつ高品質な画像を生成できます。
しかし、これを単純に連続フレームに適用すると、以下の問題が発生します：

| 問題 | 説明 | 原因 |
|------|------|------|
| **フリッカー** | フレーム間でピクセル値が急激に変動し、ちらつく | 各フレームが独立に生成されるため |
| **ドリフト** | 時間経過とともに元の内容から徐々にずれていく | 誤差が蓄積するため |
| **不整合** | 物体の形や色がフレーム間で一貫しない | 時間的な制約がないため |

この章では、これらの問題を**定量化し、解決する技術**を学びます。

In [ ]:
# ============================================================
# 合成動画データ生成：バウンシングボール
# ============================================================

def generate_bouncing_ball(num_frames=32, canvas_size=32, ball_radius=3,
                           seed=0):
    """バウンシングボールの動画を生成する

    Args:
        num_frames: フレーム数
        canvas_size: キャンバスサイズ (正方形)
        ball_radius: ボールの半径
        seed: 乱数シード
    Returns:
        (num_frames, canvas_size, canvas_size) の numpy 配列 [0, 1]
    """
    rng = np.random.RandomState(seed)
    frames = np.zeros((num_frames, canvas_size, canvas_size))

    # 初期位置と速度
    x = rng.uniform(ball_radius + 1, canvas_size - ball_radius - 1)
    y = rng.uniform(ball_radius + 1, canvas_size - ball_radius - 1)
    vx = rng.uniform(0.5, 2.0) * rng.choice([-1, 1])
    vy = rng.uniform(0.5, 2.0) * rng.choice([-1, 1])

    yy, xx = np.ogrid[:canvas_size, :canvas_size]

    for t in range(num_frames):
        # ボールを描画（滑らかな円）
        dist = np.sqrt((xx - x)**2 + (yy - y)**2)
        frames[t] = np.clip(1.0 - dist / ball_radius, 0, 1)

        # 位置更新
        x += vx
        y += vy
        # 壁で反射
        if x <= ball_radius or x >= canvas_size - ball_radius - 1:
            vx = -vx
            x = np.clip(x, ball_radius, canvas_size - ball_radius - 1)
        if y <= ball_radius or y >= canvas_size - ball_radius - 1:
            vy = -vy
            y = np.clip(y, ball_radius, canvas_size - ball_radius - 1)

    return frames


def generate_bouncing_ball_batch(num_videos=64, num_frames=32,
                                  canvas_size=32, ball_radius=3):
    """バッチでバウンシングボール動画を生成"""
    videos = []
    for i in range(num_videos):
        vid = generate_bouncing_ball(num_frames, canvas_size, ball_radius, seed=i)
        videos.append(vid)
    return np.stack(videos)  # (N, T, H, W)


# テスト生成
smooth_video = generate_bouncing_ball(num_frames=32, seed=0)
print(f'滑らかな動画の形状: {smooth_video.shape}')
print(f'値の範囲: [{smooth_video.min():.2f}, {smooth_video.max():.2f}]')

In [ ]:
# ============================================================
# フリッカー動画 vs 滑らかな動画の比較
# ============================================================

def add_flicker(video, intensity=0.3, seed=42):
    """動画にフリッカーノイズを追加する（独立ノイズで時間的不整合を模擬）"""
    rng = np.random.RandomState(seed)
    noise = rng.randn(*video.shape) * intensity
    # フレームごとに独立なブライトネス変動も追加
    brightness = 1.0 + rng.randn(video.shape[0]) * 0.2
    flickered = video * brightness[:, None, None] + noise
    return np.clip(flickered, 0, 1)

def add_drift(video, drift_rate=0.02):
    """動画にドリフト（徐々にずれる）を追加する"""
    drifted = video.copy()
    for t in range(1, len(video)):
        # 少しずつ上にシフト
        shift = int(t * drift_rate)
        if shift > 0:
            drifted[t, :-shift, :] = video[t, shift:, :]
            drifted[t, -shift:, :] = 0
    return drifted

# 3種類の動画を生成
smooth = generate_bouncing_ball(num_frames=32, seed=0)
flickered = add_flicker(smooth, intensity=0.3)
drifted = add_drift(smooth, drift_rate=0.15)

# 可視化（8フレームを選択して表示）
fig, axes = plt.subplots(3, 8, figsize=(18, 7))
frame_indices = np.linspace(0, 31, 8, dtype=int)

titles = ['滑らか（理想）', 'フリッカー（ちらつき）', 'ドリフト（漂流）']
videos = [smooth, flickered, drifted]

for row, (title, vid) in enumerate(zip(titles, videos)):
    for col, t in enumerate(frame_indices):
        axes[row, col].imshow(vid[t], cmap='viridis', vmin=0, vmax=1)
        if row == 0:
            axes[row, col].set_title(f't={t}', fontsize=10)
        axes[row, col].axis('off')
    axes[row, 0].set_ylabel(title, fontsize=12, fontweight='bold')

plt.suptitle('時間的一貫性の問題：フリッカーとドリフト', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

In [ ]:
# ============================================================
# 時間的なピクセル強度の変化を可視化
# ============================================================

fig, axes = plt.subplots(1, 3, figsize=(18, 5))

# 中心ピクセルの時間変化を追跡
cy, cx = 16, 16  # キャンバス中心

for ax, (title, vid), color in zip(
    axes,
    [('滑らか（理想）', smooth), ('フリッカー', flickered), ('ドリフト', drifted)],
    ['tab:blue', 'tab:red', 'tab:orange']
):
    # 中心付近の3x3領域の平均
    region = vid[:, cy-1:cy+2, cx-1:cx+2].mean(axis=(1, 2))
    ax.plot(range(32), region, '-o', color=color, markersize=3, linewidth=2)
    ax.set_title(title, fontsize=13, fontweight='bold')
    ax.set_xlabel('フレーム番号', fontsize=11)
    ax.set_ylabel('ピクセル強度', fontsize=11)
    ax.set_ylim(-0.1, 1.1)
    ax.grid(True, alpha=0.3)

    # フレーム間差分の標準偏差を表示
    diffs = np.diff(region)
    ax.text(0.02, 0.95, f'フレーム間差分σ={diffs.std():.4f}',
            transform=ax.transAxes, fontsize=10, va='top',
            bbox=dict(boxstyle='round', facecolor='wheat', alpha=0.5))

plt.suptitle('中心ピクセル領域の時間的強度変化', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

print('滑らかな動画：ピクセル値の変化が緩やか')
print('フリッカー：高頻度のランダムな変動が見られる')
print('ドリフト：一方向に徐々にずれていく傾向がある')

In [ ]:
# ============================================================
# フレーム間差分のヒストグラム分析
# ============================================================

fig, axes = plt.subplots(1, 3, figsize=(18, 5))

for ax, (title, vid), color in zip(
    axes,
    [('滑らか（理想）', smooth), ('フリッカー', flickered), ('ドリフト', drifted)],
    ['tab:blue', 'tab:red', 'tab:orange']
):
    # 全フレーム間差分を収集
    all_diffs = []
    for t in range(len(vid) - 1):
        diff = (vid[t + 1] - vid[t]).flatten()
        all_diffs.extend(diff.tolist())

    ax.hist(all_diffs, bins=50, color=color, alpha=0.7, density=True, edgecolor='white')
    ax.set_title(title, fontsize=13, fontweight='bold')
    ax.set_xlabel('フレーム間ピクセル差分', fontsize=11)
    ax.set_ylabel('密度', fontsize=11)
    ax.set_xlim(-0.8, 0.8)
    ax.grid(True, alpha=0.3)

    std = np.std(all_diffs)
    ax.text(0.02, 0.95, f'標準偏差={std:.4f}', transform=ax.transAxes,
            fontsize=10, va='top', bbox=dict(boxstyle='round', facecolor='wheat', alpha=0.5))

plt.suptitle('フレーム間差分の分布（ヒストグラム）', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

print('💡 滑らかな動画: 差分が0付近に集中（狭い分布）')
print('💡 フリッカー: 差分が広く分散（ノイズによる変動）')
print('💡 ドリフト: 一方向に偏った差分が含まれる')

<a id="section2"></a>
## 2. 時間的一貫性メトリクス

### 📊 一貫性をどう測るか？

動画の時間的一貫性を評価するために、いくつかの指標を定義します：

| メトリクス | 説明 | 値域 |
|-----------|------|------|
| **Frame-to-Frame MSE** | 隣接フレーム間のMSE平均 | 0〜∞（小さいほど一貫） |
| **Temporal SSIM** | 隣接フレーム間の構造類似度 | 0〜1（大きいほど一貫） |
| **Warping Error** | オプティカルフロー補正後の残差 | 0〜∞（小さいほど一貫） |
| **FVD** | 生成動画と実動画の分布距離 | 0〜∞（小さいほど良い） |

In [ ]:
# ============================================================
# 時間的一貫性メトリクスの実装
# ============================================================

def compute_temporal_consistency(video):
    """動画の時間的一貫性を計算する

    Args:
        video: numpy配列 (T, H, W) または (T, C, H, W), 値域 [0, 1]
    Returns:
        dict: 各メトリクスの値
            - frame_mse: フレーム間MSEの平均
            - frame_mse_std: フレーム間MSEの標準偏差
            - temporal_ssim: フレーム間SSIMの平均
            - smoothness: 時間方向の二次微分の大きさ（小さいほど滑らか）
    """
    if video.ndim == 4:
        # (T, C, H, W) → (T, H, W) チャンネル平均
        video = video.mean(axis=1)

    T = video.shape[0]

    # --- Frame-to-Frame MSE ---
    frame_mses = []
    for t in range(T - 1):
        mse = np.mean((video[t] - video[t + 1]) ** 2)
        frame_mses.append(mse)
    frame_mse = np.mean(frame_mses)
    frame_mse_std = np.std(frame_mses)

    # --- 簡易SSIM（構造類似度） ---
    def simple_ssim(img1, img2, C1=0.01**2, C2=0.03**2):
        """簡易的なSSIM計算（ウィンドウなし全体版）"""
        mu1, mu2 = img1.mean(), img2.mean()
        sigma1_sq = np.var(img1)
        sigma2_sq = np.var(img2)
        sigma12 = np.mean((img1 - mu1) * (img2 - mu2))
        numerator = (2 * mu1 * mu2 + C1) * (2 * sigma12 + C2)
        denominator = (mu1**2 + mu2**2 + C1) * (sigma1_sq + sigma2_sq + C2)
        return numerator / denominator

    ssims = []
    for t in range(T - 1):
        s = simple_ssim(video[t], video[t + 1])
        ssims.append(s)
    temporal_ssim = np.mean(ssims)

    # --- 時間方向の滑らかさ（二次微分） ---
    if T >= 3:
        second_deriv = []
        for t in range(1, T - 1):
            d2 = video[t - 1] - 2 * video[t] + video[t + 1]
            second_deriv.append(np.mean(d2 ** 2))
        smoothness = np.mean(second_deriv)
    else:
        smoothness = 0.0

    return {
        'frame_mse': frame_mse,
        'frame_mse_std': frame_mse_std,
        'temporal_ssim': temporal_ssim,
        'smoothness': smoothness,
    }


# 3種類の動画で計算
print('=' * 60)
print('時間的一貫性メトリクス')
print('=' * 60)

for name, vid in [('滑らか', smooth), ('フリッカー', flickered),
                   ('ドリフト', drifted)]:
    metrics = compute_temporal_consistency(vid)
    print(f'\n【{name}】')
    print(f'  Frame-to-Frame MSE:  {metrics["frame_mse"]:.6f} (±{metrics["frame_mse_std"]:.6f})')
    print(f'  Temporal SSIM:       {metrics["temporal_ssim"]:.4f}')
    print(f'  Smoothness (二次微分): {metrics["smoothness"]:.6f}')

In [ ]:
# ============================================================
# フレームごとのメトリクス推移を可視化
# ============================================================

def per_frame_metrics(video):
    """フレームごとの隣接フレーム間メトリクスを計算"""
    T = video.shape[0]
    mses = []
    for t in range(T - 1):
        mse = np.mean((video[t] - video[t + 1]) ** 2)
        mses.append(mse)
    return np.array(mses)

fig, ax = plt.subplots(figsize=(12, 5))

for name, vid, color, ls in [
    ('滑らか', smooth, 'tab:blue', '-'),
    ('フリッカー', flickered, 'tab:red', '--'),
    ('ドリフト', drifted, 'tab:orange', '-.'),
]:
    mses = per_frame_metrics(vid)
    ax.plot(range(len(mses)), mses, color=color, linestyle=ls,
            linewidth=2, marker='o', markersize=3, label=name)

ax.set_xlabel('フレーム番号 (t → t+1)', fontsize=12)
ax.set_ylabel('フレーム間 MSE', fontsize=12)
ax.set_title('フレームごとのMSE推移', fontsize=14, fontweight='bold')
ax.legend(fontsize=11)
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

print('滑らかな動画はMSEが低く安定、フリッカーは高く不安定')

### 📊 Warping Error（ワーピング誤差）の概念

Warping Errorは、オプティカルフロー（光学的流れ）を使ってフレームを整合させた後の残差です。

**手順：**
1. フレーム $t$ と $t+1$ の間のオプティカルフロー $F_{t \to t+1}$ を推定
2. フレーム $t$ をフロー $F$ に従ってワープ: $\hat{I}_{t+1} = \text{warp}(I_t, F)$
3. ワーピング誤差: $E = ||\hat{I}_{t+1} - I_{t+1}||^2$

物体が正しく追跡できていれば、ワーピング後の画像は次フレームに近くなります。
一貫性の高い動画ほど、ワーピング誤差は小さくなります。

> **注**: 本ノートブックではオプティカルフロー推定の完全な実装は省略し、
> 簡易的なピクセルシフトで概念を示します。

In [ ]:
# ============================================================
# 簡易ワーピング誤差のデモ
# ============================================================

def simple_warping_error(video):
    """簡易的なワーピング誤差（ピクセルシフト探索）

    各フレーム間で最も合致するシフト量を総当りで探索し、
    シフト補正後の残差をワーピング誤差として返す。
    """
    T = video.shape[0]
    errors = []

    for t in range(T - 1):
        frame_curr = video[t]
        frame_next = video[t + 1]
        best_err = np.mean((frame_curr - frame_next) ** 2)

        # -2〜+2ピクセルのシフトを探索
        for dy in range(-2, 3):
            for dx in range(-2, 3):
                shifted = np.roll(np.roll(frame_curr, dy, axis=0), dx, axis=1)
                err = np.mean((shifted - frame_next) ** 2)
                if err < best_err:
                    best_err = err
        errors.append(best_err)

    return np.mean(errors)

# 各動画のワーピング誤差
print('簡易ワーピング誤差:')
for name, vid in [('滑らか', smooth), ('フリッカー', flickered),
                   ('ドリフト', drifted)]:
    we = simple_warping_error(vid)
    print(f'  {name}: {we:.6f}')

### 📊 FVD（Fréchet Video Distance）の概念

FVDは、生成動画の品質を評価するための指標で、**FID（Fréchet Inception Distance）の動画版**です。

$$
\text{FVD} = ||\mu_r - \mu_g||^2 + \text{Tr}\left(\Sigma_r + \Sigma_g - 2(\Sigma_r \Sigma_g)^{1/2}\right)
$$

| 要素 | 説明 |
|------|------|
| $\mu_r, \mu_g$ | 実動画/生成動画の特徴量平均 |
| $\Sigma_r, \Sigma_g$ | 実動画/生成動画の特徴量共分散 |
| 特徴抽出器 | I3D（Inflated 3D ConvNet）を使用 |

**FIDとの違い：**
- FIDは各フレームを独立に評価 → 時間的一貫性を反映しない
- FVDはI3Dで時空間特徴を抽出 → フリッカーやドリフトも検出可能

> **注**: I3Dモデルの完全な実装は大規模なため省略しますが、
> 概念の理解のため簡易版を以下に示します。

In [ ]:
# ============================================================
# FVDの概念的な簡易実装
# ============================================================

def compute_simple_fvd(real_videos, gen_videos):
    """FVDの概念的な簡易版

    I3Dの代わりに、時空間的な統計量を特徴として使用する。
    実際のFVDはI3Dの内部特徴を使用しますが、
    ここではコンセプトの理解のため簡易的に計算します。

    Args:
        real_videos: (N, T, H, W) 実動画群
        gen_videos: (N, T, H, W) 生成動画群
    Returns:
        float: 簡易FVDスコア
    """
    def extract_features(videos):
        """簡易的な時空間特徴量を抽出"""
        features = []
        for vid in videos:
            # 各フレームの統計量
            means = vid.mean(axis=(1, 2))  # (T,)
            stds = vid.std(axis=(1, 2))    # (T,)
            # フレーム間差分
            diffs = np.diff(vid, axis=0).mean(axis=(1, 2))  # (T-1,)
            diffs = np.pad(diffs, (0, 1), constant_values=0)
            feat = np.concatenate([means, stds, diffs])
            features.append(feat)
        return np.stack(features)

    feat_r = extract_features(real_videos)
    feat_g = extract_features(gen_videos)

    # 平均と共分散を計算
    mu_r, mu_g = feat_r.mean(axis=0), feat_g.mean(axis=0)
    sigma_r = np.cov(feat_r, rowvar=False) + np.eye(feat_r.shape[1]) * 1e-6
    sigma_g = np.cov(feat_g, rowvar=False) + np.eye(feat_g.shape[1]) * 1e-6

    # Fréchet距離の計算
    diff = mu_r - mu_g
    # 行列平方根の近似 (scipy不要)
    eigvals_r = np.linalg.eigvalsh(sigma_r)
    eigvals_g = np.linalg.eigvalsh(sigma_g)
    sqrt_product_trace = np.sum(np.sqrt(np.maximum(eigvals_r * eigvals_g, 0)))

    fvd = np.sum(diff ** 2) + np.trace(sigma_r) + np.trace(sigma_g) - 2 * sqrt_product_trace
    return float(fvd)


# テスト: 実動画 vs フリッカー動画 vs ランダム動画
real_batch = generate_bouncing_ball_batch(num_videos=50, num_frames=32)
flicker_batch = np.stack([add_flicker(v, intensity=0.3, seed=i) for i, v in enumerate(real_batch)])
random_batch = np.random.rand(50, 32, 32, 32)

fvd_flicker = compute_simple_fvd(real_batch, flicker_batch)
fvd_random = compute_simple_fvd(real_batch, random_batch)
fvd_self = compute_simple_fvd(real_batch, real_batch[::-1])

print('簡易FVDスコア（小さいほど良い）:')
print(f'  実動画 vs 実動画（並び替え）: {fvd_self:.2f}')
print(f'  実動画 vs フリッカー動画:     {fvd_flicker:.2f}')
print(f'  実動画 vs ランダム動画:       {fvd_random:.2f}')
print()
print('💡 FVDが小さいほど、生成動画が実動画に近いことを示します。')

<a id="section3"></a>
## 3. Temporal Super-Resolution

### 📊 疎なキーフレームから滑らかな動画を生成

Temporal Super-Resolution（時間的超解像）は、少数の**キーフレーム**から
中間フレームを補間し、滑らかな動画を生成する技術です。

**アイデア：**
- 入力: キーフレーム $I_0, I_4, I_8, \ldots$（4フレームおき）
- 出力: 全フレーム $I_0, I_1, I_2, I_3, I_4, \ldots$（補間された中間フレーム含む）

**利点：**
- キーフレームは高品質に生成（拡散モデルなど）
- 中間フレームは軽量なCNNで高速補間
- 時間的一貫性が自然に保たれる

In [ ]:
# ============================================================
# Temporal Super-Resolution モデル
# ============================================================

class TemporalSuperResolution(nn.Module):
    """時間的超解像モデル

    2つのキーフレーム間の中間フレームをCNNで生成する。
    入力: 前後のキーフレーム (B, 2, H, W)
    出力: 補間フレーム (B, 1, H, W)
    """

    def __init__(self, hidden_dim=32):
        super().__init__()
        # エンコーダ: 2枚のキーフレームを結合して特徴抽出
        self.encoder = nn.Sequential(
            nn.Conv2d(2, hidden_dim, 3, padding=1),  # 2チャンネル（前後フレーム）
            nn.ReLU(inplace=True),
            nn.Conv2d(hidden_dim, hidden_dim, 3, padding=1),
            nn.ReLU(inplace=True),
        )
        # 時間位置の条件付け用
        self.time_embed = nn.Sequential(
            nn.Linear(1, hidden_dim),
            nn.ReLU(inplace=True),
            nn.Linear(hidden_dim, hidden_dim),
        )
        # デコーダ: 補間フレームを生成
        self.decoder = nn.Sequential(
            nn.Conv2d(hidden_dim, hidden_dim, 3, padding=1),
            nn.ReLU(inplace=True),
            nn.Conv2d(hidden_dim, hidden_dim, 3, padding=1),
            nn.ReLU(inplace=True),
            nn.Conv2d(hidden_dim, 1, 3, padding=1),
            nn.Sigmoid(),  # [0, 1]に制約
        )

    def forward(self, key_start, key_end, t_ratio):
        """補間フレームを生成

        Args:
            key_start: (B, 1, H, W) 前のキーフレーム
            key_end: (B, 1, H, W) 後のキーフレーム
            t_ratio: (B, 1) 時間的な位置 [0, 1]
                     0 = key_start, 1 = key_end
        Returns:
            (B, 1, H, W) 補間フレーム
        """
        B = key_start.shape[0]
        # 2枚を結合
        x = torch.cat([key_start, key_end], dim=1)  # (B, 2, H, W)
        feat = self.encoder(x)  # (B, hidden_dim, H, W)

        # 時間位置の埋め込み
        t_emb = self.time_embed(t_ratio)  # (B, hidden_dim)
        t_emb = t_emb.unsqueeze(-1).unsqueeze(-1)  # (B, hidden_dim, 1, 1)
        feat = feat + t_emb  # 時間位置で条件付け

        # 補間フレームを生成
        out = self.decoder(feat)  # (B, 1, H, W)
        return out


# モデルのテスト
model_tsr = TemporalSuperResolution(hidden_dim=32)
dummy_start = torch.rand(4, 1, 32, 32)
dummy_end = torch.rand(4, 1, 32, 32)
dummy_t = torch.tensor([[0.5]] * 4)
out = model_tsr(dummy_start, dummy_end, dummy_t)

total_params = sum(p.numel() for p in model_tsr.parameters())
print(f'TemporalSuperResolution:')
print(f'  入力: start={dummy_start.shape}, end={dummy_end.shape}, t={dummy_t.shape}')
print(f'  出力: {out.shape}')
print(f'  パラメータ数: {total_params:,}')

In [ ]:
# ============================================================
# 訓練データの準備（バウンシングボール）
# ============================================================

class TemporalSRDataset(Dataset):
    """Temporal Super-Resolution用のデータセット

    バウンシングボール動画からキーフレームペアと中間フレームを抽出。
    """

    def __init__(self, num_videos=200, num_frames=32, key_interval=4):
        super().__init__()
        self.key_interval = key_interval
        self.samples = []

        for i in range(num_videos):
            video = generate_bouncing_ball(num_frames, canvas_size=32, seed=i)
            # キーフレーム間の全ペアを抽出
            for start_t in range(0, num_frames - key_interval, key_interval):
                end_t = start_t + key_interval
                for mid_offset in range(1, key_interval):
                    mid_t = start_t + mid_offset
                    t_ratio = mid_offset / key_interval
                    self.samples.append((
                        torch.tensor(video[start_t], dtype=torch.float32).unsqueeze(0),
                        torch.tensor(video[end_t], dtype=torch.float32).unsqueeze(0),
                        torch.tensor(video[mid_t], dtype=torch.float32).unsqueeze(0),
                        torch.tensor([t_ratio], dtype=torch.float32),
                    ))

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):
        return self.samples[idx]


# データセット作成
dataset_tsr = TemporalSRDataset(num_videos=200, num_frames=32, key_interval=4)
print(f'データセット: {len(dataset_tsr)} サンプル')
print(f'各サンプル: start_frame, end_frame, target_frame, t_ratio')

In [ ]:
# ============================================================
# Temporal Super-Resolution の訓練
# ============================================================

model_tsr = TemporalSuperResolution(hidden_dim=32).to(device)
optimizer_tsr = torch.optim.Adam(model_tsr.parameters(), lr=1e-3)
loader_tsr = DataLoader(dataset_tsr, batch_size=64, shuffle=True)

losses_tsr = []
NUM_EPOCHS_TSR = 20

print('Temporal Super-Resolution 訓練開始...')
for epoch in range(NUM_EPOCHS_TSR):
    epoch_loss = 0.0
    count = 0
    for start_f, end_f, target_f, t_ratio in loader_tsr:
        start_f = start_f.to(device)
        end_f = end_f.to(device)
        target_f = target_f.to(device)
        t_ratio = t_ratio.to(device)

        pred = model_tsr(start_f, end_f, t_ratio)
        loss = F.mse_loss(pred, target_f)

        optimizer_tsr.zero_grad()
        loss.backward()
        optimizer_tsr.step()

        epoch_loss += loss.item() * start_f.size(0)
        count += start_f.size(0)

    avg_loss = epoch_loss / count
    losses_tsr.append(avg_loss)
    if (epoch + 1) % 5 == 0:
        print(f'  Epoch {epoch+1}/{NUM_EPOCHS_TSR}, Loss: {avg_loss:.6f}')

print('✅ 訓練完了')

In [ ]:
# ============================================================
# Temporal Super-Resolution の結果を可視化
# ============================================================

# テスト動画で補間
test_video = generate_bouncing_ball(num_frames=32, seed=999)
key_interval = 4

# キーフレームのみ抽出
key_indices = list(range(0, 32, key_interval))

# 全フレームを補間で復元
model_tsr.eval()
interpolated = np.zeros_like(test_video)

with torch.no_grad():
    for i in range(len(key_indices) - 1):
        t_start = key_indices[i]
        t_end = key_indices[i + 1]
        interpolated[t_start] = test_video[t_start]

        for mid_t in range(t_start + 1, t_end):
            t_ratio = (mid_t - t_start) / (t_end - t_start)
            start_f = torch.tensor(test_video[t_start], dtype=torch.float32).unsqueeze(0).unsqueeze(0).to(device)
            end_f = torch.tensor(test_video[t_end], dtype=torch.float32).unsqueeze(0).unsqueeze(0).to(device)
            t_r = torch.tensor([[t_ratio]], dtype=torch.float32).to(device)

            pred = model_tsr(start_f, end_f, t_r)
            interpolated[mid_t] = pred.cpu().squeeze().numpy()

    interpolated[key_indices[-1]] = test_video[key_indices[-1]]

# 線形補間も比較用に計算
linear_interp = np.zeros_like(test_video)
for i in range(len(key_indices) - 1):
    t_s = key_indices[i]
    t_e = key_indices[i + 1]
    for t in range(t_s, t_e):
        alpha = (t - t_s) / (t_e - t_s)
        linear_interp[t] = (1 - alpha) * test_video[t_s] + alpha * test_video[t_e]
linear_interp[key_indices[-1]] = test_video[key_indices[-1]]

# 可視化
fig, axes = plt.subplots(4, 8, figsize=(18, 9))
show_frames = list(range(0, 16))

row_data = [
    ('GT（正解）', test_video),
    ('キーフレームのみ', None),  # 特別処理
    ('線形補間', linear_interp),
    ('CNN補間（学習済み）', interpolated),
]

for row, (title, vid) in enumerate(row_data):
    for col, t in enumerate(show_frames):
        if row == 1:  # キーフレームのみ
            if t in key_indices:
                axes[row, col].imshow(test_video[t], cmap='viridis', vmin=0, vmax=1)
            else:
                axes[row, col].imshow(np.zeros((32, 32)), cmap='viridis', vmin=0, vmax=1)
                axes[row, col].text(16, 16, '?', ha='center', va='center',
                                     fontsize=16, color='red')
        else:
            axes[row, col].imshow(vid[t], cmap='viridis', vmin=0, vmax=1)
        if row == 0:
            marker = '★' if t in key_indices else ''
            axes[row, col].set_title(f't={t}{marker}', fontsize=9)
        axes[row, col].axis('off')
    axes[row, 0].set_ylabel(title, fontsize=11, fontweight='bold')

plt.suptitle('Temporal Super-Resolution: キーフレーム補間の比較\n（★ = キーフレーム）',
             fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

# メトリクス比較
print('\n補間品質の比較:')
for name, vid in [('線形補間', linear_interp), ('CNN補間', interpolated)]:
    mse = np.mean((vid[:16] - test_video[:16])**2)
    metrics = compute_temporal_consistency(vid)
    print(f'  {name}: MSE={mse:.6f}, SSIM={metrics["temporal_ssim"]:.4f}')

In [ ]:
# ============================================================
# 訓練損失の可視化
# ============================================================

plt.figure(figsize=(10, 5))
plt.plot(losses_tsr, 'b-o', markersize=4, linewidth=2)
plt.xlabel('エポック', fontsize=12)
plt.ylabel('損失 (MSE)', fontsize=12)
plt.title('Temporal Super-Resolution 訓練損失', fontsize=14, fontweight='bold')
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

<a id="section4"></a>
## 4. Autoregressive Video Extension

### 📊 自己回帰的な動画生成

Autoregressive（自己回帰）アプローチでは、前のフレームを条件として
次のフレームを1枚ずつ生成します。

$$
I_{t+1} = f_\theta(I_t, I_{t-1}, \ldots, I_{t-k+1})
$$

**長所：**
- 任意の長さの動画を生成可能
- 前のフレームとの一貫性が自然に保たれる

**短所：**
- 誤差が蓄積し、品質が徐々に劣化する（ドリフト問題）
- 生成速度が遅い（逐次処理）

In [ ]:
# ============================================================
# Autoregressive Video Sampler
# ============================================================

class AutoregressiveVideoSampler(nn.Module):
    """自己回帰的な動画生成モデル

    過去k枚のフレームを入力として次のフレームを予測する。
    Conv + LSTM ベースの軽量設計。
    """

    def __init__(self, canvas_size=32, hidden_dim=64, context_length=4):
        super().__init__()
        self.canvas_size = canvas_size
        self.hidden_dim = hidden_dim
        self.context_length = context_length

        # 各フレームの空間特徴抽出 (CNN)
        self.frame_encoder = nn.Sequential(
            nn.Conv2d(1, 16, 3, padding=1),
            nn.ReLU(inplace=True),
            nn.Conv2d(16, 32, 3, stride=2, padding=1),  # 32→16
            nn.ReLU(inplace=True),
            nn.Conv2d(32, 32, 3, stride=2, padding=1),  # 16→8
            nn.ReLU(inplace=True),
        )
        # 8x8x32 = 2048 → hidden_dim
        self.spatial_proj = nn.Linear(32 * 8 * 8, hidden_dim)

        # 時間的な依存関係をLSTMで捉える
        self.lstm = nn.LSTM(hidden_dim, hidden_dim, num_layers=1, batch_first=True)

        # フレームデコーダ
        self.frame_decoder = nn.Sequential(
            nn.Linear(hidden_dim, 32 * 8 * 8),
            nn.ReLU(inplace=True),
        )
        self.decoder_conv = nn.Sequential(
            nn.ConvTranspose2d(32, 32, 4, stride=2, padding=1),  # 8→16
            nn.ReLU(inplace=True),
            nn.ConvTranspose2d(32, 16, 4, stride=2, padding=1),  # 16→32
            nn.ReLU(inplace=True),
            nn.Conv2d(16, 1, 3, padding=1),
            nn.Sigmoid(),
        )

    def encode_frame(self, frame):
        """1枚のフレームを特徴量にエンコード"""
        feat = self.frame_encoder(frame)  # (B, 32, 8, 8)
        feat = feat.view(feat.size(0), -1)  # (B, 2048)
        feat = self.spatial_proj(feat)  # (B, hidden_dim)
        return feat

    def decode_feature(self, feat):
        """特徴量からフレームをデコード"""
        x = self.frame_decoder(feat)  # (B, 32*8*8)
        x = x.view(-1, 32, 8, 8)
        x = self.decoder_conv(x)  # (B, 1, 32, 32)
        return x

    def forward(self, frames):
        """次のフレームを予測

        Args:
            frames: (B, context_length, 1, H, W) 過去のフレーム列
        Returns:
            (B, 1, H, W) 予測された次のフレーム
        """
        B, T, C, H, W = frames.shape

        # 各フレームをエンコード
        feats = []
        for t in range(T):
            feat = self.encode_frame(frames[:, t])  # (B, hidden_dim)
            feats.append(feat)
        feats = torch.stack(feats, dim=1)  # (B, T, hidden_dim)

        # LSTMで時間依存を捉える
        lstm_out, _ = self.lstm(feats)  # (B, T, hidden_dim)
        last_feat = lstm_out[:, -1, :]  # (B, hidden_dim)

        # 次のフレームをデコード
        next_frame = self.decode_feature(last_feat)  # (B, 1, H, W)
        return next_frame

    @torch.no_grad()
    def generate(self, initial_frames, num_frames_to_generate):
        """初期フレームから自己回帰的に動画を生成

        Args:
            initial_frames: (1, k, 1, H, W) 初期フレーム列
            num_frames_to_generate: 生成するフレーム数
        Returns:
            (1, k + num_frames_to_generate, 1, H, W) 完全な動画
        """
        self.eval()
        frames_list = [initial_frames[:, t:t+1] for t in range(initial_frames.shape[1])]

        for _ in range(num_frames_to_generate):
            # 直近context_length枚を入力
            context = torch.cat(frames_list[-self.context_length:], dim=1)
            next_frame = self.forward(context)  # (1, 1, H, W)
            frames_list.append(next_frame.unsqueeze(1))

        all_frames = torch.cat(frames_list, dim=1)
        self.train()
        return all_frames


# モデルのテスト
model_ar = AutoregressiveVideoSampler(canvas_size=32, hidden_dim=64, context_length=4)
dummy_in = torch.rand(2, 4, 1, 32, 32)
out = model_ar(dummy_in)

total_params = sum(p.numel() for p in model_ar.parameters())
print(f'AutoregressiveVideoSampler:')
print(f'  入力: {dummy_in.shape} (B, context_length, C, H, W)')
print(f'  出力: {out.shape} (B, C, H, W)')
print(f'  パラメータ数: {total_params:,}')

In [ ]:
# ============================================================
# 自己回帰モデルの訓練データ準備
# ============================================================

class AutoregressiveDataset(Dataset):
    """自己回帰的予測用のデータセット"""

    def __init__(self, num_videos=300, num_frames=32, context_length=4):
        super().__init__()
        self.context_length = context_length
        self.samples = []

        for i in range(num_videos):
            video = generate_bouncing_ball(num_frames, canvas_size=32, seed=i)
            for t in range(num_frames - context_length):
                context = torch.tensor(
                    video[t:t + context_length],
                    dtype=torch.float32
                ).unsqueeze(1)  # (k, 1, H, W)
                target = torch.tensor(
                    video[t + context_length],
                    dtype=torch.float32
                ).unsqueeze(0)  # (1, H, W)
                self.samples.append((context, target))

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):
        return self.samples[idx]


dataset_ar = AutoregressiveDataset(num_videos=300, num_frames=32, context_length=4)
print(f'データセット: {len(dataset_ar)} サンプル')

In [ ]:
# ============================================================
# Autoregressive モデルの訓練
# ============================================================

model_ar = AutoregressiveVideoSampler(canvas_size=32, hidden_dim=64, context_length=4).to(device)
optimizer_ar = torch.optim.Adam(model_ar.parameters(), lr=1e-3)
loader_ar = DataLoader(dataset_ar, batch_size=64, shuffle=True)

losses_ar = []
NUM_EPOCHS_AR = 15

print('Autoregressive Video Sampler 訓練開始...')
for epoch in range(NUM_EPOCHS_AR):
    epoch_loss = 0.0
    count = 0
    for context, target in loader_ar:
        context = context.to(device)  # (B, k, 1, H, W)
        target = target.to(device)    # (B, 1, H, W)

        pred = model_ar(context)  # (B, 1, H, W)
        loss = F.mse_loss(pred, target)

        optimizer_ar.zero_grad()
        loss.backward()
        optimizer_ar.step()

        epoch_loss += loss.item() * context.size(0)
        count += context.size(0)

    avg_loss = epoch_loss / count
    losses_ar.append(avg_loss)
    if (epoch + 1) % 5 == 0:
        print(f'  Epoch {epoch+1}/{NUM_EPOCHS_AR}, Loss: {avg_loss:.6f}')

print('✅ 訓練完了')

In [ ]:
# ============================================================
# Autoregressive モデルの訓練損失可視化
# ============================================================

plt.figure(figsize=(10, 5))
plt.plot(losses_ar, 'r-o', markersize=4, linewidth=2)
plt.xlabel('エポック', fontsize=12)
plt.ylabel('損失 (MSE)', fontsize=12)
plt.title('Autoregressive Video Sampler 訓練損失', fontsize=14, fontweight='bold')
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

In [ ]:
# ============================================================
# 自己回帰生成と誤差蓄積の可視化
# ============================================================

# テスト動画で生成
test_video = generate_bouncing_ball(num_frames=32, seed=999)
initial = torch.tensor(test_video[:4], dtype=torch.float32).unsqueeze(0).unsqueeze(2).to(device)
# initial: (1, 4, 1, 32, 32)

model_ar.eval()
generated_ar = model_ar.generate(initial, num_frames_to_generate=28)
generated_ar = generated_ar.cpu().squeeze(2).squeeze(0).numpy()  # (32, 32, 32)

# フレームごとの誤差を計算
frame_errors = []
for t in range(32):
    err = np.mean((generated_ar[t] - test_video[t]) ** 2)
    frame_errors.append(err)

# 可視化
fig, axes = plt.subplots(3, 8, figsize=(18, 7))
show_frames = np.linspace(0, 31, 8, dtype=int)

for col, t in enumerate(show_frames):
    # GT
    axes[0, col].imshow(test_video[t], cmap='viridis', vmin=0, vmax=1)
    axes[0, col].set_title(f't={t}', fontsize=10)
    axes[0, col].axis('off')

    # 生成
    axes[1, col].imshow(generated_ar[t], cmap='viridis', vmin=0, vmax=1)
    axes[1, col].axis('off')

    # 差分
    diff = np.abs(test_video[t] - generated_ar[t])
    axes[2, col].imshow(diff, cmap='hot', vmin=0, vmax=0.5)
    axes[2, col].axis('off')

axes[0, 0].set_ylabel('GT（正解）', fontsize=11, fontweight='bold')
axes[1, 0].set_ylabel('自己回帰生成', fontsize=11, fontweight='bold')
axes[2, 0].set_ylabel('誤差（差分）', fontsize=11, fontweight='bold')

plt.suptitle('Autoregressive生成: 誤差蓄積の様子', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

In [ ]:
# ============================================================
# 誤差蓄積の定量的な分析
# ============================================================

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

# フレームごとのMSE
ax1.plot(range(32), frame_errors, 'r-o', markersize=4, linewidth=2)
ax1.axvline(x=3.5, color='gray', linestyle='--', alpha=0.5, label='初期フレーム境界')
ax1.fill_between(range(4), 0, max(frame_errors), alpha=0.1, color='green', label='初期フレーム（GT）')
ax1.set_xlabel('フレーム番号', fontsize=12)
ax1.set_ylabel('MSE', fontsize=12)
ax1.set_title('フレームごとの再構成誤差', fontsize=13, fontweight='bold')
ax1.legend(fontsize=10)
ax1.grid(True, alpha=0.3)

# 累積誤差
cumulative_errors = np.cumsum(frame_errors)
ax2.plot(range(32), cumulative_errors, 'b-o', markersize=4, linewidth=2)
ax2.set_xlabel('フレーム番号', fontsize=12)
ax2.set_ylabel('累積MSE', fontsize=12)
ax2.set_title('累積誤差の推移', fontsize=13, fontweight='bold')
ax2.grid(True, alpha=0.3)

plt.suptitle('自己回帰生成における誤差蓄積', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

print('💡 観察: フレームが進むにつれて誤差が増加しています。')
print('   これが自己回帰モデルの本質的な課題である「誤差蓄積」です。')
print(f'   初期フレーム(t=4)のMSE: {frame_errors[4]:.6f}')
print(f'   最終フレーム(t=31)のMSE: {frame_errors[31]:.6f}')
print(f'   誤差の増加倍率: {frame_errors[31] / max(frame_errors[4], 1e-8):.1f}x')

<a id="section5"></a>
## 5. 実践的テクニック

### 📊 時間的一貫性を向上させる3つの技術

実際の動画生成システムでは、以下の技術が組み合わせて使用されます：

1. **Temporal Attention**（Notebook 130で学習済み）
2. **共有ノイズスケジューリング**
3. **スライディングウィンドウ生成**

### 5.1 共有ノイズスケジューリング

動画拡散モデルでは、各フレームに**完全に独立なノイズ**を加えると
フリッカーの原因になります。

解決策: フレーム間で**ノイズの一部を共有**する

$$
\epsilon_t = \sqrt{\alpha} \cdot \epsilon_{\text{shared}} + \sqrt{1 - \alpha} \cdot \epsilon_{\text{independent}_t}
$$

- $\epsilon_{\text{shared}}$: 全フレーム共通のノイズ
- $\epsilon_{\text{independent}_t}$: フレーム固有のノイズ
- $\alpha$: 共有率（0 = 完全独立, 1 = 完全共有）

In [ ]:
# ============================================================
# 共有ノイズスケジューリングのデモ
# ============================================================

def generate_video_with_noise(video, noise_level=0.5, shared_ratio=0.0, seed=42):
    """共有ノイズ比率を変えてノイズ付き動画を生成

    Args:
        video: (T, H, W) クリーン動画
        noise_level: ノイズの強さ
        shared_ratio: ノイズの共有率 [0, 1]
        seed: 乱数シード
    Returns:
        (T, H, W) ノイズ付き動画
    """
    rng = np.random.RandomState(seed)
    T, H, W = video.shape

    # 共有ノイズ（全フレーム共通）
    shared_noise = rng.randn(H, W)
    # 独立ノイズ（フレームごと）
    indep_noise = rng.randn(T, H, W)

    # 混合
    alpha = shared_ratio
    mixed_noise = (np.sqrt(alpha) * shared_noise[None, :, :] +
                   np.sqrt(1 - alpha) * indep_noise) * noise_level

    return np.clip(video + mixed_noise, 0, 1)


# 共有率を変えて比較
test_vid = generate_bouncing_ball(num_frames=16, seed=0)
shared_ratios = [0.0, 0.3, 0.6, 0.9]

fig, axes = plt.subplots(len(shared_ratios) + 1, 8, figsize=(18, 3 * (len(shared_ratios) + 1)))
show_t = np.linspace(0, 15, 8, dtype=int)

# GTを表示
for col, t in enumerate(show_t):
    axes[0, col].imshow(test_vid[t], cmap='viridis', vmin=0, vmax=1)
    axes[0, col].set_title(f't={t}', fontsize=10)
    axes[0, col].axis('off')
axes[0, 0].set_ylabel('GT', fontsize=11, fontweight='bold')

# 各共有率
for row, alpha in enumerate(shared_ratios):
    noisy = generate_video_with_noise(test_vid, noise_level=0.5, shared_ratio=alpha)
    metrics = compute_temporal_consistency(noisy)
    for col, t in enumerate(show_t):
        axes[row + 1, col].imshow(noisy[t], cmap='viridis', vmin=0, vmax=1)
        axes[row + 1, col].axis('off')
    label = f'共有率={alpha:.0%}\nSSIM={metrics["temporal_ssim"]:.3f}'
    axes[row + 1, 0].set_ylabel(label, fontsize=10, fontweight='bold')

plt.suptitle('共有ノイズ比率の効果（noise_level=0.5）', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

print('💡 共有率が高いほどフレーム間の一貫性（SSIM）が向上します。')
print('   ただし共有率が高すぎると動きが失われるため、適切なバランスが重要です。')

### 5.2 スライディングウィンドウ生成

長い動画を一度に生成するのはメモリ的に困難です。
**スライディングウィンドウ**方式では、短いクリップを重複させながら逐次生成します。

```
ウィンドウ1: [F0, F1, F2, F3, F4, F5, F6, F7]
ウィンドウ2:         [F4, F5, F6, F7, F8, F9, F10, F11]
ウィンドウ3:                 [F8, F9, F10, F11, F12, F13, F14, F15]
                     ^^^^^^^^^^^^
                     重複部分（条件付けに使用）
```

重複部分のフレームを次のウィンドウの「条件」として使うことで、
接合部分の一貫性を保ちます。

In [ ]:
# ============================================================
# スライディングウィンドウ生成のデモ
# ============================================================

def sliding_window_generation(model, initial_frames, total_frames=32,
                               window_size=8, overlap=4):
    """スライディングウィンドウ方式で長い動画を生成

    Args:
        model: 自己回帰モデル
        initial_frames: (1, k, 1, H, W) 初期フレーム
        total_frames: 目標フレーム数
        window_size: ウィンドウサイズ
        overlap: 重複フレーム数
    Returns:
        (total_frames, H, W) 生成動画
    """
    model.eval()
    k = initial_frames.shape[1]
    all_frames = [initial_frames[0, t, 0].cpu().numpy() for t in range(k)]

    while len(all_frames) < total_frames:
        # 直近のコンテキストから生成
        context_start = max(0, len(all_frames) - model.context_length)
        context_frames = np.stack(all_frames[context_start:])
        context_tensor = torch.tensor(context_frames, dtype=torch.float32)
        context_tensor = context_tensor.unsqueeze(0).unsqueeze(2).to(device)

        # コンテキスト長を調整
        if context_tensor.shape[1] > model.context_length:
            context_tensor = context_tensor[:, -model.context_length:]

        with torch.no_grad():
            next_frame = model(context_tensor)
        all_frames.append(next_frame.cpu().squeeze().numpy())

    return np.stack(all_frames[:total_frames])


# スライディングウィンドウ生成
test_vid = generate_bouncing_ball(num_frames=32, seed=999)
init_frames = torch.tensor(test_vid[:4], dtype=torch.float32).unsqueeze(0).unsqueeze(2).to(device)

sw_video = sliding_window_generation(model_ar, init_frames, total_frames=32,
                                      window_size=8, overlap=4)

# 可視化
fig, axes = plt.subplots(2, 8, figsize=(18, 5))
show_t = np.linspace(0, 31, 8, dtype=int)

for col, t in enumerate(show_t):
    axes[0, col].imshow(test_vid[t], cmap='viridis', vmin=0, vmax=1)
    axes[0, col].set_title(f't={t}', fontsize=10)
    axes[0, col].axis('off')

    axes[1, col].imshow(sw_video[t], cmap='viridis', vmin=0, vmax=1)
    axes[1, col].axis('off')

axes[0, 0].set_ylabel('GT（正解）', fontsize=11, fontweight='bold')
axes[1, 0].set_ylabel('SW生成', fontsize=11, fontweight='bold')

plt.suptitle('スライディングウィンドウ方式による長尺動画生成', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

### 5.3 Temporal Attention（復習）

Notebook 130で学んだTemporal Attentionは、時間的一貫性の最も強力な手法の一つです。

**要点の復習：**
- 同じ空間位置の異なるフレーム間でAttentionを計算
- `(B, T, C, H, W)` → `(B*H*W, T, C)` にreshapeして計算
- 因果マスクで自己回帰生成にも対応
- Spatial + Temporal の分離で計算量を大幅に削減

```
Spatial Attention:  各フレーム内のパッチ間の関係
Temporal Attention: 同じ位置の異なるフレーム間の関係
→ 組み合わせることで時空間的な一貫性を学習
```

In [ ]:
# ============================================================
# Temporal Attentionによる一貫性向上の概念図
# ============================================================

def visualize_temporal_attention_consistency():
    """Temporal Attentionが一貫性を保つ仕組みの概念図"""
    fig, axes = plt.subplots(1, 2, figsize=(16, 5))

    # 左: Temporal Attentionなし（独立生成）
    ax = axes[0]
    T = 8
    for t in range(T):
        np.random.seed(t * 10)
        y_offset = 0
        x_center = t * 2
        # 各フレーム独立なので位置がばらつく
        ball_y = 3 + np.random.randn() * 0.5
        circle = plt.Circle((x_center, ball_y), 0.4, color='steelblue', alpha=0.7)
        ax.add_patch(circle)
        ax.text(x_center, -0.5, f't={t}', ha='center', fontsize=9)

    ax.set_xlim(-1, T * 2)
    ax.set_ylim(-1.5, 6)
    ax.set_aspect('equal')
    ax.set_title('Temporal Attentionなし\n（各フレーム独立 → 位置がばらつく）',
                 fontsize=12, fontweight='bold')
    ax.axhline(y=3, color='gray', linestyle='--', alpha=0.3)
    ax.axis('off')

    # 右: Temporal Attentionあり（一貫した軌道）
    ax = axes[1]
    for t in range(T):
        x_center = t * 2
        ball_y = 3 + 0.8 * np.sin(t * 0.5)  # 滑らかな軌道
        circle = plt.Circle((x_center, ball_y), 0.4, color='coral', alpha=0.7)
        ax.add_patch(circle)
        ax.text(x_center, -0.5, f't={t}', ha='center', fontsize=9)
        if t > 0:
            prev_y = 3 + 0.8 * np.sin((t - 1) * 0.5)
            ax.annotate('', xy=(x_center, ball_y), xytext=((t-1)*2, prev_y),
                        arrowprops=dict(arrowstyle='->', color='gray', lw=1.5, alpha=0.5))

    ax.set_xlim(-1, T * 2)
    ax.set_ylim(-1.5, 6)
    ax.set_aspect('equal')
    ax.set_title('Temporal Attentionあり\n（フレーム間を参照 → 滑らかな軌道）',
                 fontsize=12, fontweight='bold')
    ax.axhline(y=3, color='gray', linestyle='--', alpha=0.3)
    ax.axis('off')

    plt.suptitle('Temporal Attentionが時間的一貫性を保つ仕組み',
                 fontsize=14, fontweight='bold')
    plt.tight_layout()
    plt.show()

visualize_temporal_attention_consistency()
print('Temporal Attentionにより、各フレームが他のフレームを参照して')
print('一貫した特徴（位置、色、形状）を維持できます。')

<a id="section6"></a>
## 6. 比較実験

### 📊 手法間の一貫性メトリクス比較

これまでに実装した各手法の時間的一貫性を定量的に比較します。

In [ ]:
# ============================================================
# 各手法の一貫性メトリクスを比較
# ============================================================

# テスト動画群で評価
num_test = 20
results = {
    'GT（正解）': [],
    'フリッカー付き': [],
    '線形補間': [],
    'CNN補間 (TSR)': [],
    '自己回帰 (AR)': [],
}

model_tsr.eval()
model_ar.eval()

for seed in range(num_test):
    gt_vid = generate_bouncing_ball(num_frames=32, seed=1000 + seed)

    # GT
    results['GT（正解）'].append(compute_temporal_consistency(gt_vid))

    # フリッカー
    flick = add_flicker(gt_vid, intensity=0.3, seed=seed)
    results['フリッカー付き'].append(compute_temporal_consistency(flick))

    # 線形補間
    key_idx = list(range(0, 32, 4))
    lin = np.zeros_like(gt_vid)
    for i in range(len(key_idx) - 1):
        ts, te = key_idx[i], key_idx[i + 1]
        for tt in range(ts, te):
            a = (tt - ts) / (te - ts)
            lin[tt] = (1 - a) * gt_vid[ts] + a * gt_vid[te]
    lin[key_idx[-1]] = gt_vid[key_idx[-1]]
    results['線形補間'].append(compute_temporal_consistency(lin))

    # CNN補間 (TSR)
    interp = np.zeros_like(gt_vid)
    with torch.no_grad():
        for i in range(len(key_idx) - 1):
            ts, te = key_idx[i], key_idx[i + 1]
            interp[ts] = gt_vid[ts]
            for mid in range(ts + 1, te):
                tr = (mid - ts) / (te - ts)
                sf = torch.tensor(gt_vid[ts], dtype=torch.float32).view(1, 1, 32, 32).to(device)
                ef = torch.tensor(gt_vid[te], dtype=torch.float32).view(1, 1, 32, 32).to(device)
                tv = torch.tensor([[tr]], dtype=torch.float32).to(device)
                p = model_tsr(sf, ef, tv)
                interp[mid] = p.cpu().squeeze().numpy()
        interp[key_idx[-1]] = gt_vid[key_idx[-1]]
    results['CNN補間 (TSR)'].append(compute_temporal_consistency(interp))

    # 自己回帰 (AR)
    init = torch.tensor(gt_vid[:4], dtype=torch.float32).unsqueeze(0).unsqueeze(2).to(device)
    gen_ar = model_ar.generate(init, num_frames_to_generate=28)
    gen_ar_np = gen_ar.cpu().squeeze(2).squeeze(0).numpy()
    results['自己回帰 (AR)'].append(compute_temporal_consistency(gen_ar_np))

# 結果を集計
print('=' * 70)
print(f'{" 時間的一貫性メトリクス比較 ":=^60}')
print('=' * 70)
print(f'{"手法":<20} {"Frame MSE":>12} {"Temporal SSIM":>15} {"Smoothness":>12}')
print('-' * 70)

summary = {}
for method, metric_list in results.items():
    avg_mse = np.mean([m['frame_mse'] for m in metric_list])
    avg_ssim = np.mean([m['temporal_ssim'] for m in metric_list])
    avg_smooth = np.mean([m['smoothness'] for m in metric_list])
    summary[method] = {'frame_mse': avg_mse, 'temporal_ssim': avg_ssim, 'smoothness': avg_smooth}
    print(f'{method:<20} {avg_mse:>12.6f} {avg_ssim:>15.4f} {avg_smooth:>12.6f}')

In [ ]:
# ============================================================
# メトリクス比較の棒グラフ
# ============================================================

methods = list(summary.keys())
colors = ['tab:blue', 'tab:red', 'tab:orange', 'tab:green', 'tab:purple']

fig, axes = plt.subplots(1, 3, figsize=(18, 6))
x = np.arange(len(methods))
width = 0.6

# Frame MSE（小さいほど良い）
vals = [summary[m]['frame_mse'] for m in methods]
bars = axes[0].bar(x, vals, width, color=colors)
axes[0].set_title('Frame-to-Frame MSE\n（小さいほど良い ↓）', fontsize=13, fontweight='bold')
axes[0].set_ylabel('MSE', fontsize=11)
axes[0].set_xticks(x)
axes[0].set_xticklabels(methods, fontsize=9, rotation=20, ha='right')
axes[0].grid(True, alpha=0.3, axis='y')
for bar, val in zip(bars, vals):
    axes[0].text(bar.get_x() + bar.get_width() / 2, bar.get_height(),
                 f'{val:.4f}', ha='center', va='bottom', fontsize=9)

# Temporal SSIM（大きいほど良い）
vals = [summary[m]['temporal_ssim'] for m in methods]
bars = axes[1].bar(x, vals, width, color=colors)
axes[1].set_title('Temporal SSIM\n（大きいほど良い ↑）', fontsize=13, fontweight='bold')
axes[1].set_ylabel('SSIM', fontsize=11)
axes[1].set_xticks(x)
axes[1].set_xticklabels(methods, fontsize=9, rotation=20, ha='right')
axes[1].set_ylim(0, 1.05)
axes[1].grid(True, alpha=0.3, axis='y')
for bar, val in zip(bars, vals):
    axes[1].text(bar.get_x() + bar.get_width() / 2, bar.get_height(),
                 f'{val:.3f}', ha='center', va='bottom', fontsize=9)

# Smoothness（小さいほど良い）
vals = [summary[m]['smoothness'] for m in methods]
bars = axes[2].bar(x, vals, width, color=colors)
axes[2].set_title('Smoothness (二次微分)\n（小さいほど良い ↓）', fontsize=13, fontweight='bold')
axes[2].set_ylabel('二次微分の大きさ', fontsize=11)
axes[2].set_xticks(x)
axes[2].set_xticklabels(methods, fontsize=9, rotation=20, ha='right')
axes[2].grid(True, alpha=0.3, axis='y')
for bar, val in zip(bars, vals):
    axes[2].text(bar.get_x() + bar.get_width() / 2, bar.get_height(),
                 f'{val:.4f}', ha='center', va='bottom', fontsize=9)

plt.suptitle('手法別の時間的一貫性メトリクス比較', fontsize=15, fontweight='bold')
plt.tight_layout()
plt.show()

In [ ]:
# ============================================================
# レーダーチャートによる手法比較
# ============================================================

def radar_chart(summary_dict):
    """各手法の特性をレーダーチャートで比較"""
    # メトリクスを正規化（全て「高いほど良い」に統一）
    methods = list(summary_dict.keys())
    metrics_names = ['時間的一貫性\n(1-MSE)', 'SSIM', '滑らかさ\n(1-Smooth)']

    # 正規化用の最大値
    max_mse = max(s['frame_mse'] for s in summary_dict.values())
    max_smooth = max(s['smoothness'] for s in summary_dict.values())

    values_dict = {}
    for method in methods:
        s = summary_dict[method]
        values_dict[method] = [
            1.0 - s['frame_mse'] / max(max_mse, 1e-8),  # MSEが小さい→スコア高
            s['temporal_ssim'],
            1.0 - s['smoothness'] / max(max_smooth, 1e-8),
        ]

    # レーダーチャート
    N = len(metrics_names)
    angles = np.linspace(0, 2 * np.pi, N, endpoint=False).tolist()
    angles += angles[:1]  # 閉じる

    fig, ax = plt.subplots(figsize=(8, 8), subplot_kw=dict(polar=True))
    colors = ['tab:blue', 'tab:red', 'tab:orange', 'tab:green', 'tab:purple']

    for i, (method, vals) in enumerate(values_dict.items()):
        vals_closed = vals + vals[:1]
        ax.plot(angles, vals_closed, '-o', color=colors[i % len(colors)],
                linewidth=2, markersize=6, label=method)
        ax.fill(angles, vals_closed, color=colors[i % len(colors)], alpha=0.1)

    ax.set_xticks(angles[:-1])
    ax.set_xticklabels(metrics_names, fontsize=11)
    ax.set_ylim(0, 1.05)
    ax.set_title('手法別の時間的一貫性\nレーダーチャート', fontsize=14,
                 fontweight='bold', pad=20)
    ax.legend(loc='upper right', bbox_to_anchor=(1.3, 1.1), fontsize=9)
    plt.tight_layout()
    plt.show()

radar_chart(summary)

In [ ]:
# ============================================================
# 全手法のサンプル動画を並べて比較
# ============================================================

# 1つのテスト動画に対する各手法の出力を可視化
test_vid = generate_bouncing_ball(num_frames=32, seed=1000)
key_idx = list(range(0, 32, 4))

# 線形補間
lin_vid = np.zeros_like(test_vid)
for i in range(len(key_idx) - 1):
    ts, te = key_idx[i], key_idx[i + 1]
    for t in range(ts, te):
        a = (t - ts) / (te - ts)
        lin_vid[t] = (1 - a) * test_vid[ts] + a * test_vid[te]
lin_vid[key_idx[-1]] = test_vid[key_idx[-1]]

# CNN補間
cnn_vid = np.zeros_like(test_vid)
with torch.no_grad():
    for i in range(len(key_idx) - 1):
        ts, te = key_idx[i], key_idx[i + 1]
        cnn_vid[ts] = test_vid[ts]
        for mid in range(ts + 1, te):
            tr = (mid - ts) / (te - ts)
            sf = torch.tensor(test_vid[ts], dtype=torch.float32).view(1, 1, 32, 32).to(device)
            ef = torch.tensor(test_vid[te], dtype=torch.float32).view(1, 1, 32, 32).to(device)
            tv = torch.tensor([[tr]], dtype=torch.float32).to(device)
            p = model_tsr(sf, ef, tv)
            cnn_vid[mid] = p.cpu().squeeze().numpy()
    cnn_vid[key_idx[-1]] = test_vid[key_idx[-1]]

# AR生成
init_f = torch.tensor(test_vid[:4], dtype=torch.float32).unsqueeze(0).unsqueeze(2).to(device)
ar_vid = model_ar.generate(init_f, num_frames_to_generate=28)
ar_vid = ar_vid.cpu().squeeze(2).squeeze(0).numpy()

# 全手法を並べて表示（6フレーム）
fig, axes = plt.subplots(4, 6, figsize=(16, 11))
show_t = [0, 5, 10, 15, 20, 25]

for col, t in enumerate(show_t):
    axes[0, col].imshow(test_vid[t], cmap='viridis', vmin=0, vmax=1)
    axes[0, col].set_title(f't={t}', fontsize=10)
    axes[1, col].imshow(lin_vid[t], cmap='viridis', vmin=0, vmax=1)
    axes[2, col].imshow(cnn_vid[t], cmap='viridis', vmin=0, vmax=1)
    axes[3, col].imshow(np.clip(ar_vid[t], 0, 1), cmap='viridis', vmin=0, vmax=1)
    for row in range(4):
        axes[row, col].axis('off')

axes[0, 0].set_ylabel('GT', fontsize=12, fontweight='bold')
axes[1, 0].set_ylabel('線形補間', fontsize=12, fontweight='bold')
axes[2, 0].set_ylabel('CNN補間\n(TSR)', fontsize=12, fontweight='bold')
axes[3, 0].set_ylabel('自己回帰\n(AR)', fontsize=12, fontweight='bold')

plt.suptitle('全手法のサンプル動画比較', fontsize=15, fontweight='bold')
plt.tight_layout()
plt.show()

### 📊 手法の特徴まとめ

| 手法 | 一貫性 | 品質 | 速度 | 長尺対応 | 主な用途 |
|------|--------|------|------|----------|----------|
| **Temporal Super-Resolution** | ◎ | ○ | 速い | △ | キーフレーム補間 |
| **Autoregressive** | ○ | △（劣化あり） | 遅い | ◎ | 無限長動画 |
| **Temporal Attention** | ◎ | ◎ | 中程度 | ○ | 一括生成 |
| **共有ノイズ** | ○ | ○ | 速い | ◎ | 拡散モデル全般 |
| **スライディングウィンドウ** | ○ | ○ | 中程度 | ◎ | 長尺動画 |

実際の最先端システム（Sora, Runway Gen-3など）では、
これらの技術を**組み合わせて**使用しています。

<a id="summary"></a>
## 7. まとめと自己評価クイズ

### 🎯 このノートブックで学んだこと

**時間的一貫性の問題**
- フリッカー：フレーム間のランダムな変動
- ドリフト：時間経過に伴う累積的なずれ
- これらを定量化するメトリクス（MSE, SSIM, Warping Error, FVD）

**解決手法**
- ✓ `compute_temporal_consistency`: 一貫性メトリクスの実装
- ✓ `TemporalSuperResolution`: キーフレーム間のCNN補間
- ✓ `AutoregressiveVideoSampler`: LSTM+CNNによる逐次生成
- ✓ 共有ノイズスケジューリングの効果
- ✓ スライディングウィンドウによる長尺対応

### 📊 チートシート

| 概念 | 数式/コード | 説明 |
|------|------------|------|
| Frame MSE | `mean((f[t] - f[t+1])^2)` | 隣接フレーム差 |
| Temporal SSIM | `(2μ₁μ₂+C₁)(2σ₁₂+C₂) / ...` | 構造類似度 |
| 共有ノイズ | `√α·ε_shared + √(1-α)·ε_indep` | 一貫性向上 |
| FVD | `‖μ_r-μ_g‖² + Tr(Σ_r+Σ_g-2√(Σ_rΣ_g))` | 動画品質指標 |

### ⚠️ よくあるエラー

#### エラー #1: 自己回帰生成で品質が急速に劣化する

```python
# ❌ 問題: 生成フレームをそのまま次の入力に使う → 誤差が蓄積
for t in range(100):
    next_frame = model(context)
    context = update(context, next_frame)  # 誤差が雪だるま式に増大

# ✅ 対策: 定期的にキーフレームでリセット / 温度パラメータを下げる
for t in range(100):
    if t % 16 == 0:
        next_frame = diffusion_model.sample(condition=keyframe)  # リセット
    else:
        next_frame = model(context)
```

#### エラー #2: 共有ノイズ比率が高すぎて動きがなくなる

```python
# ❌ 共有率100% → 全フレーム同一（静止画）
noise = shared_noise  # alpha=1.0

# ✅ 適切な共有率（0.3〜0.7程度）
alpha = 0.5
noise = sqrt(alpha) * shared + sqrt(1-alpha) * independent
```

#### エラー #3: Temporal SSIMの計算で値域を間違える

```python
# ❌ 画像の値域が[-1, 1]なのにSSIMの定数が[0, 1]前提
C1 = 0.01**2  # [0, 1]用

# ✅ 値域に合わせて定数を調整
L = 2.0  # [-1, 1]の場合のダイナミックレンジ
C1 = (0.01 * L)**2
```

### 🎓 自己評価クイズ

### Q1: フリッカーとドリフトの違いを説明してください。

<details>
<summary>💡 答えを見る</summary>

**答え:**

- **フリッカー（ちらつき）**: フレーム間でピクセル値が**ランダムに**急激変動する現象。
  各フレームが独立に生成される場合に発生し、高周波ノイズとして現れます。
- **ドリフト（漂流）**: 時間経過とともに内容が**一方向に**徐々にずれていく現象。
  自己回帰生成での誤差蓄積が主な原因で、低周波の変動として現れます。

対策も異なり、フリッカーには共有ノイズやTemporal Attention、
ドリフトにはキーフレームリセットやアンカリングが有効です。

</details>

---

### Q2: FVDとFIDの違いは何ですか？なぜ動画評価にFVDが必要ですか？

<details>
<summary>💡 答えを見る</summary>

**答え:**

| | FID | FVD |
|---|---|---|
| 対象 | 画像 | 動画 |
| 特徴抽出 | Inception V3 | I3D (Inflated 3D ConvNet) |
| 入力 | 1枚の画像 | T枚のフレーム列 |
| 時間情報 | 無し | 有り |

FIDは各フレームを独立に評価するため、フリッカーやドリフトといった
**時間的な不整合**を検出できません。FVDはI3Dで時空間特徴を抽出するため、
動画固有の品質問題も評価できます。

</details>

---

### Q3: Temporal Super-Resolutionの入力と出力は何ですか？

<details>
<summary>💡 答えを見る</summary>

**答え:**

- **入力**: 2つのキーフレーム $I_{t_1}$, $I_{t_2}$ と時間位置比率 $r \in [0, 1]$
- **出力**: 補間されたフレーム $\hat{I}_{t_1 + r \cdot (t_2 - t_1)}$

CNNがキーフレーム間の非線形な動きを学習し、
単純な線形補間よりも高品質なフレームを生成します。
$r = 0$ ならkey_start、$r = 1$ ならkey_endに近い出力となります。

</details>

---

### Q4: 自己回帰的動画生成の最大の課題と、その対策を2つ挙げてください。

<details>
<summary>💡 答えを見る</summary>

**答え:**

最大の課題: **誤差蓄積（Error Accumulation）**
- 1フレームの小さな誤差が次フレームの入力に含まれ、雪だるま式に増大

対策:
1. **キーフレームリセット**: 一定間隔で高品質なキーフレームを生成し直し、
   誤差蓄積をリセットする
2. **スライディングウィンドウ**: 短いクリップを重複させながら生成し、
   重複部分の整合性を保つ
3. (追加) **Scheduled Sampling**: 訓練時に予測フレームをランダムに入力に混ぜ、
   誤差耐性を向上させる

</details>

---

### Q5: 共有ノイズスケジューリングで $\alpha = 0$ と $\alpha = 1$ はそれぞれ何を意味しますか？

<details>
<summary>💡 答えを見る</summary>

**答え:**

- **$\alpha = 0$（共有率0%）**: 全フレームに完全に独立なノイズが加わる。
  拡散過程の各フレームは他のフレームと無関係にノイズ化される。
  → 逆拡散時にフリッカーが発生しやすい

- **$\alpha = 1$（共有率100%）**: 全フレームに同一のノイズが加わる。
  拡散過程で全フレームが同じノイズパターンを持つ。
  → 逆拡散時に全フレームが同一画像に収束（静止画になる）

適切な値は $\alpha = 0.3 \sim 0.7$ 程度で、
時間的一貫性と動きの多様性のバランスを取ります。

</details>

---

### ✅ 学習チェックリスト

- [ ] フリッカーとドリフトの違いを説明でき、合成データで再現できる
- [ ] `compute_temporal_consistency` で一貫性メトリクスを計算できる
- [ ] FVDの概念（I3D特徴 + Fréchet距離）を説明できる
- [ ] `TemporalSuperResolution` でキーフレーム補間を実装・訓練できる
- [ ] `AutoregressiveVideoSampler` で逐次生成し、誤差蓄積を確認できる
- [ ] 共有ノイズスケジューリングの効果を説明できる
- [ ] 各手法の一貫性メトリクスを比較分析できる

---

**次のステップ**: Notebook 135 では、これらの技術を組み合わせた
**制御可能な動画生成**（テキスト条件、動き条件など）を学びます！